# CHAS — What a Tabulation Loses

D05 met a number that is an estimate by definition and a rule in practice. This notebook meets
the other half of the same estate: a dataset that is a survey by origin and a count by
presentation.

Each year the Census Bureau retabulates American Community Survey microdata to HUD's
specification. The result is the **CHAS** — Comprehensive Housing Affordability Strategy —
data, and local governments use it to plan how HUD funds are spent. Every figure in it
descends from the ACS.

The ACS's defining feature is that it publishes a margin of error on everything. CHAS serves
**129 count columns and no margin on any of them.**

That is not an error, and §1 shows CHAS and the ACS agree closely on the quantities they
share. It is a *property of derivation*: a tabulation carries the values forward and leaves
the uncertainty behind, and the people planning against the result see a hard count where the
survey underneath said roughly this, give or take a quarter.

**The regime — the custom tabulation.** Distinct from the five in the handbook, and from D05's
administrative number, in four ways this notebook measures:

- **The uncertainty does not survive** the tabulation (§1).
- **It runs a full release behind** its own source, and the API's default vintage moves (§2).
- **Disclosure suppression arrives as `null`**, with no flag and no count (§3).
- **The documentation is not the data**: columns that are documented and never served,
  dictionary entries that contradict the values, and an index that repeats itself (§4).

## Learning Objectives

1. **Quantify what a derived product discards** by recovering the source's published
   uncertainty for the same quantity, rather than noting its absence in prose.
2. **Distinguish agreement from equivalence** — two figures can match and still not be the
   same kind of claim.
3. **Match reference periods before comparing**, when a derived product lags its source.
4. **Handle suppression that arrives as an absence** rather than as a flag or a sentinel.
5. **Test documentation against data**, and treat the data as the authority when they
   disagree.

## Prerequisites

**Knowledge.** D03 (margins of error, and why an estimate without one asserts more than it
knows), D27 (ACS `B25070`, rent burden, and the "not computed" category), D05 (HAMFI and
administratively determined numbers — CHAS's income buckets are cut against it).

**Credentials.** `CENSUS_API_KEY` and `HUD_API_KEY`.

## Data Provenance

| Source | What | Regime | Vintage |
|---|---|---|---|
| HUD CHAS API, pinned county extract | Custom ACS tabulation, 133 Virginia counties | **Custom tabulation** | `2018-2022`, explicit |\n| HUD CHAS API, live | The county index, and the vintage probe in §2 | **Custom tabulation** | Explicit per call |
| Census ACS 5-Year 2022, `B25070` | Gross rent as a share of household income | Margins of error | Dataset year, matched to CHAS |
| Census ACS 5-Year 2023, `B25070` | The same, one release later | Margins of error | Dataset year |

CHAS's current release is built on **ACS 2018–2022**, so the ACS comparison uses the **2022**
5-year file. Comparing it against ACS 2023 — the release D27 uses — would attribute three
years of housing-market change to the tabulation. §2 measures that difference rather than
assuming it away.

In [1]:
"""Configuration."""
from __future__ import annotations

import re

import numpy as np
import pandas as pd

from pathlib import Path
from datetime import date

import sys
from pathlib import Path

#: Find the package by walking UP from wherever this runs, so the notebook works
#: whether it is executed in place or from the repository root. Sixteen notebooks
#: imported `krl_tutorials_data` with no path discovery and therefore ran only
#: from the root -- which nobody noticed while every run happened to start there.
_repo = next((c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (c / "src" / "krl_tutorials_data").is_dir()), None)
if _repo is None:
    raise SystemExit("krl_tutorials_data not found above " + str(Path.cwd()))
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo / "src"))

from krl_tutorials_data.connectors import BulkFileSource, frame_from_bulk_file
from krl_tutorials_data.sources import (
    CHAS_DOCUMENTED_NOT_SERVED,
    CHAS_MISDOCUMENTED,
    EmptyResultError,
    fetch_acs_wide,
    fetch_chas,
    fetch_chas_county_list,
)

pd.set_option("display.width", 125)
pd.set_option("display.max_columns", 30)

STATE = 51                 # Virginia: D09 and D15 also work here, so the geography is shared
CHAS_VINTAGE = "2018-2022" # explicit; the API's default is "latest" and moves
ACS_MATCHED = 2022         # the ACS release CHAS 2018-2022 is built from
ACS_CURRENT = 2023         # the latest ACS release, used only to measure the lag

#: CHAS columns this notebook uses, with the meaning taken from HUD's published
#: dictionary rather than inferred. Naming the target is D01's lesson.
CHAS_COLUMNS = {
    "A17": "total renter-occupied households",
    "I16": "renters, cost burden over 30 percent",
    "I17": "renters, cost burden over 50 percent",   # documented, never served -- see section 4
    "D2":  "renters, cost burden 30 percent or less",
    "D5":  "renters, cost burden over 30 to 50 percent",
    "D8":  "renters, cost burden over 50 percent",
    "D11": "renters, cost burden not available",
}

#: ACS B25070 brackets that are 30 percent of income or more.
BURDENED_BRACKETS = (7, 8, 9, 10)

print(f"CHAS {CHAS_VINTAGE} against ACS {ACS_MATCHED} 5-year, state FIPS {STATE}")

CHAS 2018-2022 against ACS 2022 5-year, state FIPS 51


In [2]:
"""Load the pinned county extract, and show the index defect it was built around."""
#: Anchored to the repository root rather than the working directory.
#: These two notebooks resolved `data/hud/...` relative to cwd, so they
#: ran only from the repo root -- the same dependence that put sixteen
#: provenance records in a stray folder.
REPO_ROOT = next(c for c in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                 if (c / "src" / "krl_tutorials_data").is_dir())

CHAS_EXTRACT = BulkFileSource(
    path=REPO_ROOT / Path("data/hud/chas") / f"VA_{CHAS_VINTAGE}_county.csv",
    publisher="U.S. Department of Housing and Urban Development (HUD USER)",
    dataset=f"CHAS county summary level, {CHAS_VINTAGE}, state FIPS {STATE}",
    vintage=CHAS_VINTAGE,
    retrieved_on=date(2026, 9, 14),
    sha256="2ea228f3bdcbb32b2df1fc48f98cb307ee08a9e6d5a60e9958ed3dc7b17383b5",
    terms_url="https://www.huduser.gov/portal/about/pdr_data_license.html",
    notes=(
        "One API request per county; HUD paces at 60 per minute, so a state sweep "
        "takes minutes and is pinned rather than repeated on every execution.",
        "Suppressed cells are JSON null in the API and empty in this CSV.",
    ),
)

chas, chas_prov = frame_from_bulk_file(
    CHAS_EXTRACT,
    lambda path: pd.read_csv(path, dtype={"entityId": str}),
    expected_columns={"entityId": "unpadded HUD county entity id", "A17": "total renters"},
)
coded = [c for c in chas.columns if re.fullmatch(r"[A-Z]+[0-9]+", c)]
print(f"CHAS counties: {len(chas)}   coded columns: {len(coded)}   pin verified")

# The index this was built from, fetched live because it is small and it is a finding.
counties = fetch_chas_county_list(state_id=STATE)
print(
    f"\nHUD's county index serves {counties.attrs['served_rows']} rows for "
    f"{len(counties)} distinct counties; {counties.attrs['duplicate_rows']} are duplicates."
)
print(
    "  Iterating it as served fetches every county twice. A frame built that way\n"
    "  double-counts every household in the state, and nothing about it looks wrong."
)
no_rows = REPO_ROOT / Path("data/hud/chas") / f"VA_{CHAS_VINTAGE}_no_rows.txt"
listed_but_empty = no_rows.read_text().split() if no_rows.exists() else []
print(
    f"\n  entity ids the index lists and the API serves no rows for: {listed_but_empty}"
    f"\n  (515 is Bedford city, which ceased to be an independent city in 2013 and is"
    f"\n   still listed.)"
)

CHAS counties: 133   coded columns: 129   pin verified



HUD's county index serves 267 rows for 134 distinct counties; 133 are duplicates.
  Iterating it as served fetches every county twice. A frame built that way
  double-counts every household in the state, and nothing about it looks wrong.

  entity ids the index lists and the API serves no rows for: ['515']
  (515 is Bedford city, which ceased to be an independent city in 2013 and is
   still listed.)


## 1. The uncertainty does not survive the tabulation

CHAS reports how many renter households pay more than 30% of income for housing. So does the
ACS, in `B25070`, from the same microdata — and the ACS publishes a margin on it.

Recovering that margin is the only way to see what CHAS's users are not shown.

In [3]:
"""The same quantity from both sources, with the ACS margin carried."""
variables = {f"B25070_{i:03d}E": f"bracket_{i}" for i in range(1, 12)}
variables.update({f"B25070_{i:03d}M": f"bracket_{i}_moe" for i in range(1, 12)})

acs, acs_prov = fetch_acs_wide(
    year=ACS_MATCHED, variables=variables, geography="county:*", within=f"state:{STATE}"
)
estimate = acs[[f"B25070_{i:03d}E" for i in BURDENED_BRACKETS]].sum(axis=1)
# Margins combine in quadrature, not by addition (D03).
margin = np.sqrt((acs[[f"B25070_{i:03d}M" for i in BURDENED_BRACKETS]] ** 2).sum(axis=1))
acs = acs.assign(
    entityId=acs["county"].astype(str).str.lstrip("0"),
    acs_burdened=estimate,
    acs_margin=margin,
)

paired = chas.merge(
    acs[["entityId", "acs_burdened", "acs_margin", "NAME"]], on="entityId", how="inner"
).dropna(subset=["I16", "acs_burdened"])
print(f"counties on both sides: {len(paired)} of {len(chas)} CHAS rows and {len(acs)} ACS counties")

relative = 100 * paired.acs_margin / paired.acs_burdened.replace(0, np.nan)
print(
    f"\nACS margin on this quantity, as a share of the estimate:"
    f"\n  median {relative.median():.1f}%   p90 {relative.quantile(0.90):.1f}%"
    f"   max {relative.max():.1f}%"
)
coded = [c for c in chas.columns if re.fullmatch(r"[A-Z]+[0-9]+", c)]
print(f"CHAS margin columns: 0 of {len(coded)}.")

counties on both sides: 133 of 133 CHAS rows and 133 ACS counties

ACS margin on this quantity, as a share of the estimate:
  median 24.6%   p90 46.0%   max 101.7%
CHAS margin columns: 0 of 129.


In [4]:
"""Agreement is not equivalence: check whether CHAS and the ACS actually differ."""
difference = 100 * (paired.I16 - paired.acs_burdened) / paired.acs_burdened.replace(0, np.nan)
inside = int((abs(paired.I16 - paired.acs_burdened) <= paired.acs_margin).sum())
print(
    f"CHAS against the ACS on the same quantity, same reference period:"
    f"\n  median difference {difference.median():+.1f}%"
    f"\n  CHAS falls inside the ACS 90% margin in {inside} of {len(paired)} counties"
    f" ({100 * inside / len(paired):.0f}%)"
)
print(
    "\nSo CHAS is not a different answer. It is the same answer with the interval removed."
    "\nThat is the finding: the disagreement to report is not between the numbers, it is"
    "\nbetween what each one claims to know."
)

widest = paired.assign(pct=relative).nlargest(5, "pct")[
    ["NAME", "I16", "acs_burdened", "acs_margin", "pct"]
]
print("\ncounties where the discarded interval is widest:")
print(widest.to_string(index=False, formatters={"pct": "{:.0f}%".format}))

CHAS against the ACS on the same quantity, same reference period:
  median difference -3.5%
  CHAS falls inside the ACS 90% margin in 130 of 133 counties (98%)

So CHAS is not a different answer. It is the same answer with the interval removed.
That is the finding: the disagreement to report is not between the numbers, it is
between what each one claims to know.

counties where the discarded interval is widest:
                           NAME   I16  acs_burdened  acs_margin  pct
         Craig County, Virginia  87.0          87.0   88.447725 102%
      Highland County, Virginia  44.0          44.0   38.379682  87%
         Bland County, Virginia  50.0          50.0   39.572718  79%
Northumberland County, Virginia 144.0         147.0  109.526253  75%
          Bath County, Virginia 104.0         107.0   73.205191  68%


A planner allocating funds on a CHAS count in the median county here is working from a figure
the underlying survey could not pin down to better than about a quarter of its own size. The
count is not wrong. It is simply not the kind of object it appears to be, and the appearance
is created by the format rather than by anyone's claim.

**[rule]** Where a derived product drops its source's uncertainty, recover the source's
interval and report it alongside. A missing margin is not a small margin.

## 2. A release behind, and a default that moves

In [5]:
"""What the API serves, what it documents, and what it does when you do not ask."""
print(f"CHAS latest vintage in use here: {CHAS_VINTAGE} (an ACS 2018-2022 tabulation)")
print(f"Latest ACS 5-year release available: {ACS_CURRENT}")
print(
    "  HUD's own API documentation lists year ranges only up to '2014-2018', while"
    "\n  2016-2020, 2017-2021 and 2018-2022 are all served. The documentation is not"
    "\n  the authority on which vintages exist; the API is."
)
try:
    fetch_chas(year="2019-2023", summary_level="county", state_id=STATE, entity_id=59)
    print("\n  2019-2023 IS served")
except EmptyResultError:
    print("\n  2019-2023 returns HTTP 200 and no rows: CHAS is one full ACS release behind.")

# How much does three years of lag matter for this quantity?
acs_now, _ = fetch_acs_wide(
    year=ACS_CURRENT, variables=variables, geography="county:*", within=f"state:{STATE}"
)
now = acs_now[[f"B25070_{i:03d}E" for i in BURDENED_BRACKETS]].sum(axis=1)
acs_now = acs_now.assign(entityId=acs_now["county"].astype(str).str.lstrip("0"), now=now)
lagged = paired.merge(acs_now[["entityId", "now"]], on="entityId", how="inner")
drift = 100 * (lagged.now - lagged.acs_burdened) / lagged.acs_burdened.replace(0, np.nan)
print(
    f"\nSame quantity, ACS {ACS_MATCHED} vs ACS {ACS_CURRENT}, {len(lagged)} counties:"
    f"\n  median change {drift.median():+.1f}%   p10 {drift.quantile(0.10):+.1f}%"
    f"   p90 {drift.quantile(0.90):+.1f}%"
    f"\n  counties moving more than 10% either way: "
    f"{int((drift.abs() > 10).sum())} of {len(lagged)}"
)
print(
    "\n  That is the cost of comparing CHAS against the CURRENT ACS instead of the one it"
    "\n  was built from. It would be attributed to the tabulation, and it is the calendar."
)

CHAS latest vintage in use here: 2018-2022 (an ACS 2018-2022 tabulation)
Latest ACS 5-year release available: 2023
  HUD's own API documentation lists year ranges only up to '2014-2018', while
  2016-2020, 2017-2021 and 2018-2022 are all served. The documentation is not
  the authority on which vintages exist; the API is.



  2019-2023 returns HTTP 200 and no rows: CHAS is one full ACS release behind.



Same quantity, ACS 2022 vs ACS 2023, 133 counties:
  median change -0.7%   p10 -18.0%   p90 +16.9%
  counties moving more than 10% either way: 53 of 133

  That is the cost of comparing CHAS against the CURRENT ACS instead of the one it
  was built from. It would be attributed to the tabulation, and it is the calendar.


## 3. A category that was retired and kept its column

Twenty of the 129 columns come back empty. The obvious reading is disclosure suppression —
HUD's release note for these tables does mention "enhanced disclosure avoidance protections…
resulting in some previously-available estimates being suppressed."

That reading is wrong, and the data says so before any documentation is consulted.

In [6]:
"""Empty columns: are they suppressed cells, or something else?

Disclosure suppression responds to cell size -- it withholds a value where too few
households sit behind it. So it must VARY: small counties lose cells that large ones
keep. A column that is empty in every county of every size is not responding to
anything, and is therefore not suppression.
"""
empty = chas[coded].isna().sum()
always = sorted(empty[empty == len(chas)].index, key=lambda c: (c[0], int(c[1:])))
sometimes = sorted(empty[(empty > 0) & (empty < len(chas))].index)

print(f"coded cells {chas[coded].size}   empty {int(empty.sum())} ({100 * empty.sum() / chas[coded].size:.1f}%)")
print(f"  columns empty in EVERY one of the {len(chas)} counties: {len(always)}")
print(f"  columns empty in SOME counties but not others: {len(sometimes)}")
print(f"  {len(chas)} counties x {len(always)} columns = {len(chas) * len(always)}")
print(
    "\n  Every empty cell is accounted for by whole columns, and NOT ONE column varies"
    "\n  between counties. Suppression that never responds to county size is not"
    "\n  suppression."
)
print(f"\n  the columns: {always}")

coded cells 17157   empty 2660 (15.5%)
  columns empty in EVERY one of the 133 counties: 20
  columns empty in SOME counties but not others: 0
  133 counties x 20 columns = 2660

  Every empty cell is accounted for by whole columns, and NOT ONE column varies
  between counties. Suppression that never responds to county size is not
  suppression.

  the columns: ['B7', 'B8', 'B9', 'E3', 'E7', 'E11', 'E15', 'E19', 'E23', 'F3', 'F7', 'F11', 'F15', 'F19', 'F23', 'G3', 'G7', 'G11', 'G15', 'G19']


In [7]:
"""What those columns have in common, from HUD's own dictionary."""
RETIRED_CATEGORY = "cost burden not available, no other problems"
print(f"All {len(always)} belong to one family of measures:\n")
print("  B7, B8, B9          'Cost burden not available, no other problems', by tenure")
print("  E3 E7 E11 E15 E19 E23   the same, crossed with income, owners and renters")
print("  F3 F7 F11 F15 F19 F23   the same, renters only")
print("  G3 G7 G11 G15 G19       the same, owners only")
print(
    f"\nAnd HUD's release note for these tables says that category was not suppressed."
    f"\nIt was COMBINED: 'The \"Cost burden cannot be computed, none of the above"
    f"\nproblems\" housing unit problems category is now combined with the \"Has none of"
    f"\nthe 4 housing problems\" category.'"
)
print(
    "\nSo the category was retired, its households were folded into the neighbouring"
    "\ncategory, and the column stayed. The dictionary still documents it, the API still"
    "\nserves it, and it is empty everywhere and always."
)

# What it costs: the neighbouring category absorbed those households.
kept = ["B4", "B5", "B6"]
print(
    f"\n  A reader summing 'none of the problems' plus 'cost burden not available' to"
    f"\n  rebuild a total gets NaN, or -- if they fill it with zero -- exactly the same"
    f"\n  number as 'none of the problems' alone. Both are right, for the wrong reason:"
    f"\n  the households are already inside B4-B6."
)
print(f"\n  B4+B5 vs B6 (owners + renters vs total) agrees in "
      f"{int((chas.B4 + chas.B5 == chas.B6).sum())} of {len(chas)} counties, so the"
      f" surviving\n  categories do close among themselves.")

All 20 belong to one family of measures:

  B7, B8, B9          'Cost burden not available, no other problems', by tenure
  E3 E7 E11 E15 E19 E23   the same, crossed with income, owners and renters
  F3 F7 F11 F15 F19 F23   the same, renters only
  G3 G7 G11 G15 G19       the same, owners only

And HUD's release note for these tables says that category was not suppressed.
It was COMBINED: 'The "Cost burden cannot be computed, none of the above
problems" housing unit problems category is now combined with the "Has none of
the 4 housing problems" category.'

So the category was retired, its households were folded into the neighbouring
category, and the column stayed. The dictionary still documents it, the API still
serves it, and it is empty everywhere and always.

  A reader summing 'none of the problems' plus 'cost burden not available' to
  rebuild a total gets NaN, or -- if they fill it with zero -- exactly the same
  number as 'none of the problems' alone. Both are right, for the wr

This is a different defect from D09's suppression and a worse one to meet unprepared. QCEW
withholds a value and flags it; the analyst who reads the flag is safe. Here nothing is
withheld — the measure no longer exists — and the artifact that remains is a column name in a
dictionary, an empty field in every response, and no note in either.

**[rule]** An always-empty column is a schema claim, not a data gap. Check whether a column's
emptiness *varies* before calling it suppression: if it does not vary, it is not responding to
the data, and the explanation is in the publisher's release notes rather than in its cells.

**[rule]** `fetch_chas` keeps an empty cell as `NaN`, never `0`. Here that is what makes the
retired category visible instead of silently contributing zeros to a total.

## 4. The documentation is not the data

Every check in this section was prompted by reading HUD's published dictionary against a
response. None required any statistics.

In [8]:
"""Documented columns that are never served."""
served = set(coded)
absent = sorted(CHAS_DOCUMENTED_NOT_SERVED & set(CHAS_COLUMNS) | CHAS_DOCUMENTED_NOT_SERVED)
print(f"documented in HUD's dictionary but never served: {absent}")
print(f"  present in this response: {sorted(CHAS_DOCUMENTED_NOT_SERVED & served) or 'none'}")
print(
    "\n  This notebook wanted I17 — renters with cost burden over 50 percent — and it is one"
    "\n  of them. The quantity is reachable another way (D8), which is the only reason the"
    "\n  analysis survives. Code written from the dictionary alone would raise a KeyError on"
    "\n  a column name taken from the publisher's own documentation."
)

documented in HUD's dictionary but never served: ['G21', 'G22', 'G23', 'H17', 'I17', 'J17']
  present in this response: none

  This notebook wanted I17 — renters with cost burden over 50 percent — and it is one
  of them. The quantity is reachable another way (D8), which is the only reason the
  analysis survives. Code written from the dictionary alone would raise a KeyError on
  a column name taken from the publisher's own documentation.


In [9]:
"""Dictionary entries whose descriptions are duplicated, tested against the values.

HUD documents H13 and H14 identically: 'Household Income >100% HAMFI (Owners and
Renters) and Cost burden > 30%'. Identical descriptions imply identical quantities,
so the columns must be equal. The surrounding pattern (H1/H2, H4/H5, H7/H8, H10/H11)
alternates '>30%' and '>50%', which implies the second is a subset instead.

Only the data can settle it.
"""
for second, first in sorted(CHAS_MISDOCUMENTED.items()):
    pair = chas[[first, second]].dropna()
    equal = int((pair[second] == pair[first]).sum())
    smaller = int((pair[second] < pair[first]).sum())
    larger = int((pair[second] > pair[first]).sum())
    verdict = (
        "identical, as documented" if equal == len(pair)
        else "DISTINCT quantities, documented identically"
    )
    print(
        f"  {second} vs {first} over {len(pair)} counties: equal {equal}, "
        f"smaller {smaller}, larger {larger}  ->  {verdict}"
    )
print(
    "\n  The second column is never larger and is usually smaller, which is what a subset"
    "\n  looks like and is not what a duplicate looks like. The dictionary is wrong, and"
    "\n  anyone trusting it reports a 'greater than 50 percent' figure as 'greater than 30'."
)

  H14 vs H13 over 133 counties: equal 1, smaller 132, larger 0  ->  DISTINCT quantities, documented identically
  I14 vs I13 over 133 counties: equal 46, smaller 87, larger 0  ->  DISTINCT quantities, documented identically
  J14 vs J13 over 133 counties: equal 3, smaller 130, larger 0  ->  DISTINCT quantities, documented identically

  The second column is never larger and is usually smaller, which is what a subset
  looks like and is not what a duplicate looks like. The dictionary is wrong, and
  anyone trusting it reports a 'greater than 50 percent' figure as 'greater than 30'.


And two more, already handled in `fetch_chas` rather than left for the reader:

- **A zero-padded entity id returns nothing.** `entityId=59` is Fairfax County; `entityId=059`
  returns HTTP 200 and `[]`. The same publisher's FMR API requires the *padded* ten-character
  form, so the two HUD APIs take incompatible spellings of the same county and neither refuses
  the wrong one. `fetch_chas` strips the padding and raises `EmptyResultError` on an empty
  body, because an empty frame is not an empty population.
- **The county index repeats itself**, as §0 reported. `fetch_chas_county_list` deduplicates
  and reports how many it removed.

**[rule]** When documentation and data disagree, the data is the authority and the
disagreement is a finding. Record it; do not quietly code around it.

## 5. The partition that does not close

D27 established that ACS rent-burden brackets fall short of their published universe because
a "not computed" category sits outside them. CHAS carries the same category — `D11`, "cost
burden not available" — so the same check applies to a different source.

In [10]:
"""Do the renter cost-burden categories account for every renter household?"""
parts = ["D2", "D5", "D8", "D11"]
closure = chas[parts + ["A17", "geoname"]].dropna()
summed = closure[parts].sum(axis=1)
gap = summed - closure.A17
exact = int((gap.abs() < 0.5).sum())
print(f"counties where D2+D5+D8+D11 equals A17 (total renters): {exact} of {len(closure)}")
print(
    f"  gap as a share of the renter total: median {100 * (gap / closure.A17.replace(0, np.nan)).median():+.2f}%"
    f"   largest over-count {100 * (gap / closure.A17.replace(0, np.nan)).max():+.1f}%"
    f"   largest short-fall {100 * (gap / closure.A17.replace(0, np.nan)).min():+.1f}%"
)
not_available = 100 * closure.D11 / closure.A17.replace(0, np.nan)
print(
    f"\n  'cost burden not available' as a share of renters: median {not_available.median():.1f}%,"
    f" max {not_available.max():.1f}%"
)
print(
    "\n  The gaps are rounding: CHAS counts are published to the nearest 5. The categories"
    "\n  DO account for every renter, which is the opposite of what D27 found in the ACS's"
    "\n  own brackets -- and worth reporting as such rather than assuming the defect"
    "\n  travels. 'Cost burden not available' is real but small here, so dropping it"
    "\n  would bias a burden rate in the same direction as D27 and far less."
)

counties where D2+D5+D8+D11 equals A17 (total renters): 35 of 133
  gap as a share of the renter total: median +0.00%   largest over-count +2.9%   largest short-fall -1.1%

  'cost burden not available' as a share of renters: median 1.6%, max 13.1%

  The gaps are rounding: CHAS counts are published to the nearest 5. The categories
  DO account for every renter, which is the opposite of what D27 found in the ACS's
  own brackets -- and worth reporting as such rather than assuming the defect
  travels. 'Cost burden not available' is real but small here, so dropping it
  would bias a burden rate in the same direction as D27 and far less.


In [11]:
"""FINDINGS"""
print("FINDINGS")
print("=" * 78)
print(f"""
1. The uncertainty does not survive the tabulation. CHAS serves {len(coded)} count
   columns and no margin on any of them. On the one quantity both publish, the
   ACS margin runs a median {relative.median():.1f}% of the estimate (p90 {relative.quantile(0.90):.1f}%,
   max {relative.max():.1f}%). CHAS is not a different answer: it falls inside that
   margin in {inside} of {len(paired)} counties, median difference {difference.median():+.1f}%.
   It is the same estimate with the interval removed.
2. CHAS is one full ACS release behind. Its latest vintage is a {CHAS_VINTAGE}
   tabulation while ACS {ACS_CURRENT} is published, and 2019-2023 returns HTTP 200 with no
   rows. Over that gap the same quantity moved a median {drift.median():+.1f}%, and more
   than 10% either way in {int((drift.abs() > 10).sum())} of {len(lagged)} counties -- which a comparison against
   the current ACS would charge to the tabulation rather than to the calendar.
3. A retired category kept its column. {int(empty.sum())} of {chas[coded].size} coded cells are empty,
   and every one is accounted for by {len(always)} columns that are empty in ALL {len(chas)}
   counties, with {len(sometimes)} columns varying between counties. Suppression responds to
   cell size and so must vary; this does not. All {len(always)} are the "cost burden not
   available, no other problems" family, which HUD's release note says was
   COMBINED into the neighbouring category rather than suppressed. The measure
   was retired, the households moved, and the column, the dictionary entry and
   the empty field all remain.
4. The documentation is not the data, in three ways. {len(sorted(CHAS_DOCUMENTED_NOT_SERVED))} columns are
   documented and never served, including the one this notebook first reached for.
   {len(CHAS_MISDOCUMENTED)} dictionary entries carry a duplicate description and the values contradict
   it. And HUD's county index served {counties.attrs['served_rows']} rows for {len(counties)} counties.
5. A zero-padded entity id returns HTTP 200 and no rows, while the same
   publisher's FMR API requires the padded form. Two APIs, one publisher, one
   county, incompatible spellings, and neither refuses the wrong one.
6. The cost-burden categories nearly close: exactly in {exact} of {len(closure)} counties,
   with a median gap of {100 * (gap / closure.A17.replace(0, np.nan)).median():+.2f}% and a worst case of {100 * (gap / closure.A17.replace(0, np.nan)).abs().max():.1f}% -- rounding,
   not a missing category. 'Cost burden not available' is a median {not_available.median():.1f}% of
   renters here against the {13.4}% D27 measured for the ACS's own "not computed",
   so the D27 denominator hazard EXISTS in CHAS but is an order of magnitude
   smaller. Reported because the check is what establishes that, not the guess.
""".rstrip())
print("=" * 78)

FINDINGS

1. The uncertainty does not survive the tabulation. CHAS serves 129 count
   columns and no margin on any of them. On the one quantity both publish, the
   ACS margin runs a median 24.6% of the estimate (p90 46.0%,
   max 101.7%). CHAS is not a different answer: it falls inside that
   margin in 130 of 133 counties, median difference -3.5%.
   It is the same estimate with the interval removed.
2. CHAS is one full ACS release behind. Its latest vintage is a 2018-2022
   tabulation while ACS 2023 is published, and 2019-2023 returns HTTP 200 with no
   rows. Over that gap the same quantity moved a median -0.7%, and more
   than 10% either way in 53 of 133 counties -- which a comparison against
   the current ACS would charge to the tabulation rather than to the calendar.
3. A retired category kept its column. 2660 of 17157 coded cells are empty,
   and every one is accounted for by 20 columns that are empty in ALL 133
   counties, with 0 columns varying between counties. Suppres

## Limitations

- **One state.** Every figure is Virginia. The suppression rate, the index duplication and the
  dictionary errors are properties of the publisher and should generalise; the *magnitudes*
  in §1 and §2 are not claimed to.
- **One quantity for the margin comparison.** §1 uses renter cost burden over 30%, because it
  is the quantity CHAS and `B25070` most nearly share. A different column could carry a
  different margin, and this does not establish a corpus-wide figure.
- **The correspondence is close, not exact.** CHAS and `B25070` are built from the same
  microdata but not by the same recipe — CHAS's universe excludes some households `B25070`
  retains. The comparison is reported as agreement within a margin, never as an identity.
- **The lag measurement is ACS-to-ACS.** §2 measures how much the *quantity* moved between ACS
  releases, which bounds the cost of a mismatched comparison. It does not establish what CHAS
  would say for 2019–2023, which is unpublished.
- **HAMFI is not examined.** CHAS's income categories are cut against HUD's Area Median Family
  Income, an administratively determined figure of the kind D05 covers. Whether those
  boundaries moved between vintages is exercise 3.

## Key Takeaways

- **A derived product can keep the value and lose the warrant.** Recover the source's interval
  and report it; a missing margin is not a small margin.
- **Agreement is not equivalence.** CHAS matches the ACS and still makes a different kind of
  claim. Check what a number asserts, not only what it says.
- **Match the reference period before comparing**, or the calendar gets charged to the method.
- **Check whether emptiness varies before calling it suppression.** A column empty in every
  geography is not responding to the data; the explanation is in the release notes.
- **Test the documentation against the data.** Three defects here were found by reading the
  dictionary beside a response, and the data settled every one.
- **An index can lie by repeating itself.** A duplicated listing double-counts a whole state
  and nothing looks wrong.

## Exercises

1. **Does it generalise?** Repeat §3 and §4 for three more states. Are the suppression rate and
   the index duplication properties of the publisher, as claimed, or of Virginia?
2. **Bound what was suppressed.** CHAS publishes totals (`A17`, `A18`) alongside suppressed
   components. For each county, how tightly do the surviving cells bound the missing ones, and
   is the bound ever narrow enough to be useful?
3. **The moving boundary.** CHAS income categories are cut at percentages of HAMFI, which HUD
   re-sets annually. Retrieve HAMFI for two CHAS vintages and determine how much of the change
   in "households at or below 30% HAMFI" is households moving and how much is the line moving.
   This is a Kitagawa decomposition (D07) with an administrative boundary as one component.
4. **Propagate what CHAS dropped.** Using the ACS margins from §1, put an interval on a CHAS
   count and re-rank the counties. How many rank positions are identified, in the sense of D16?
5. **The other tabulations.** CHAS tables 1–12 exist beyond the API's flat column set. Retrieve
   one from the bulk download and check whether the margins are absent there too, or whether
   the API is what drops them.
6. **Write the rule.** Draft the paragraph this corpus should use before adopting any custom
   tabulation: what must be established about its source, its lag, its suppression and its
   documentation before a figure from it is reported. Test it against this notebook and D05.

## References

1. **U.S. Department of Housing and Urban Development.** *Consolidated Planning/CHAS Data.*
   <https://www.huduser.gov/portal/datasets/cp.html>
2. **HUD USER.** *CHAS Dataset API Documentation* — the data dictionary tested in §4.
   <https://www.huduser.gov/portal/dataset/chas-api.html>
3. **U.S. Census Bureau.** *American Community Survey 5-Year Estimates*, table `B25070`.
4. **U.S. Census Bureau.** *Understanding and Using ACS Data*, Appendix: Measures of Error —
   the quadrature rule used in §1.
5. **Data Source Handbook**, `docs/methodology.md` — §1 for the regimes this
   source is distinguished from, §3 for denominators and partitions.